[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/15_decision_guide_and_use_cases.ipynb)

# Decision guide & use cases

> **Fine-tuning series — 15 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. Conceptual/reference — no GPU setup needed.


## Which to pick

**How-axis (capacity vs. cost):**
- *Full FT* — large data + real domain shift + you have the GPUs.
- *LoRA* — the default; small/medium data, base fits in bf16.
- *QLoRA* — same as LoRA but the base won't fit in 16-bit (big model / small GPU).
- *Prompt tuning* — extreme parameter thrift / many lightweight task variants; lower ceiling.
- *DoRA* — near drop-in LoRA upgrade; usually higher quality for one flag.
- *IA³ / BitFit / adapters* — even thriftier baselines; IA³ is the most parameter-efficient.
- *(Unsloth)* — bolt onto LoRA/QLoRA for speed + memory, no downside on supported models.

**What-axis (objective):**
- *Instruction tuning (SFT)* — teach formats, tasks, tone, domain knowledge. Start here.
- *DPO* — cheaply align to human/AI preferences after SFT; no reward model.
- *GRPO* — optimize a measurable reward (correctness, format); reasoning models.
- *PPO* — classic RLHF with a reward model; most complex, rarely the first choice now.
- *ORPO / KTO* — cheaper alignment: ORPO needs no reference model or SFT pass; KTO needs only thumbs up/down, no pairs.
- *Reward modeling* — train the scalar RM that PPO / best-of-N consume.
- *Rejection sampling / distillation* — bootstrap from generated data (best-of-N → SFT; or imitate a bigger teacher).

**Typical real stack:** SFT (instruction tuning) with **QLoRA**, accelerated by
**Unsloth**, then **DPO** (also QLoRA) for alignment.


## Use cases — who should use which technique

You pick **one method from each axis** and combine them (how params update + what
objective); Unsloth optionally accelerates either. This section maps real situations
and roles to that combination.

### By situation (constraint → pick)

| Your situation | Use | Why |
|---|---|---|
| One consumer GPU (≤24 GB), model ≥ 7B | **QLoRA** (+ Unsloth) | 4-bit base is the only way it fits; Unsloth ~2× faster, ~½ VRAM |
| Multi-GPU + large labeled data + real domain shift | **Full fine-tuning** | enough compute to justify it; LoRA may cap quality here |
| Base fits in bf16, want fast iteration | **LoRA** (or **DoRA**) | cheap, near-full quality; DoRA buys quality for one flag |
| Many per-customer / per-task variants from one base | **LoRA adapters** (hot-swap) | one frozen base + tiny swappable adapters, not N checkpoints |
| Tiny storage / hundreds of tasks | **IA³** or **prompt tuning** | smallest footprint; accept a lower ceiling |
| Abundant *unlabeled* domain data, few labels | **Self-supervised continued pretraining → SFT** | adapt representations first so scarce labels go further |
| Need instruction-following / a format / a tone | **Instruction tuning (SFT)** | the starting point; teaches the task itself |
| Have preference pairs (chosen/rejected) | **DPO** (+ LoRA) | aligns cheaply, no reward model or sampling loop |
| Have only thumbs up/down (unpaired) | **KTO** | works on binary, unpaired feedback |
| Want SFT + preference in one pass, no reference model | **ORPO** | single stage, reference-free |
| Can *score* outputs programmatically (correctness/format) | **GRPO** or **RFT** | reward-driven; GRPO is the reasoning-model recipe |
| Need a scalar scorer for PPO / best-of-N | **Reward modeling** | trains the RM those methods consume |
| Want a small fast model that mimics a big one | **Distillation (GKD)** / teacher→SFT | compress capability into a cheaper model |
| Facts change often / need citations | **RAG (not fine-tuning)** | retrieve at inference; don't bake facts into weights |

### By persona (the full stack)

- **Solo dev / Kaggler on Colab** → QLoRA + Unsloth for instruction tuning; add DPO later if you collect preferences.
- **SaaS serving many clients** → instruction-tune once, ship a small **LoRA adapter per client** on one shared base.
- **Researcher chasing reasoning SOTA** → SFT warm-start, then **GRPO** with a verifiable reward; DoRA if staying PEFT.
- **Enterprise brand chatbot** → **SFT** to the brand voice, then **DPO/KTO** from user feedback for alignment.
- **Medical / legal domain specialist** → **continued self-supervised pretraining** on the unlabeled domain corpus → **QLoRA SFT** on the scarce labels; use RAG for fast-changing facts.
- **On-device / edge deployment** → **distill** a big teacher into a small student, then QLoRA-tune and quantize.

### When *not* to fine-tune

If you mainly need fresh or citable **facts**, reach for **RAG or tools** first — fine-tuning
bakes knowledge in and ages badly. If a few prompt examples already work, that's
**in-context learning**, not fine-tuning. Fine-tune when you need new *behavior, format, or
skill*, lower latency/cost at scale, or alignment that prompting can't reach.